In [ ]:
import os
import pandas as pd

def parse_accel_filename(filename):
    name = os.path.splitext(filename)[0]
    parts = name.split('_')
    return {
        'topology': parts[0],
        'leak_type': parts[1],
        'flow': parts[2],
        'sensor': parts[3],
    }

def build_accel_index_recursive(root_folder):
    rows = []
    for dirpath, _, filenames in os.walk(root_folder):
        for fname in filenames:
            if fname.endswith('.csv'):
                meta = parse_accel_filename(fname)
                meta['filename'] = fname
                meta['file_path'] = os.path.join(dirpath, fname)
                meta['modality'] = 'accelerometer'
                rows.append(meta)
    return pd.DataFrame(rows)

df_accel = build_accel_index_recursive(r'C:\Users\layan\Downloads\leaks\leaks_acc')
df_accel['is_leak'] = (df_accel['leak_type'] != 'NL').astype(int)

print(df_accel.shape)
print(df_accel['leak_type'].value_counts())
print(df_accel['is_leak'].value_counts())
print(df_accel.groupby(['topology', 'leak_type']).size())

In [ ]:
import os
import pandas as pd

folder = r'C:\Users\layan\Downloads\leaks\leaks_acc'

rows = []
for dirpath, _, filenames in os.walk(folder):
    for fname in filenames:
        if fname.endswith('.csv'):
            name = os.path.splitext(fname)[0]
            parts = name.split('_')
            print(f"File: {fname} | Parts: {parts} | Count: {len(parts)}")
            rows.append({
                'topology': parts[0],
                'leak_type': parts[1],
                'flow': parts[2],
                'sensor': parts[3],
                'filename': fname,
                'file_path': os.path.join(dirpath, fname),
            })
            if len(rows) >= 5:
                break
    if len(rows) >= 5:
        break

df_test = pd.DataFrame(rows)
print("\nShape:", df_test.shape)
print("Columns:", df_test.columns.tolist())
print(df_test)

In [ ]:
import numpy as np
import pandas as pd

WINDOW_S = 2          # 2-second windows
STRIDE_S = 1          # 50% overlap

def make_windows(signal, sr, window_s=WINDOW_S, stride_s=STRIDE_S):
    """Split a signal into overlapping windows. Returns array of shape (n_windows, window_samples)."""
    win_len = int(window_s * sr)
    stride = int(stride_s * sr)
    n_windows = (len(signal) - win_len) // stride + 1
    windows = np.zeros((n_windows, win_len), dtype=signal.dtype)
    for i in range(n_windows):
        start = i * stride
        windows[i] = signal[start:start + win_len]
    return windows

def build_windowed_index(df_meta, loader_fn, sr, window_s=WINDOW_S, stride_s=STRIDE_S):
    """For each signal in df_meta, load it, window it, and return a long DataFrame
    where each row is one window with parent metadata."""
    rows = []
    all_windows = []
    for idx, meta_row in df_meta.iterrows():
        sig = loader_fn(meta_row['file_path'])
        wins = make_windows(sig, sr, window_s, stride_s)
        for w_idx, w in enumerate(wins):
            row = meta_row.to_dict()
            row['window_idx'] = w_idx
            row['signal_id'] = idx  # CRITICAL: groups windows by parent signal
            rows.append(row)
            all_windows.append(w)
    return pd.DataFrame(rows), np.array(all_windows)

In [ ]:

df_accel_win, accel_windows = build_windowed_index(df_accel, load_accel, SR_ACCEL)

print("Accel windows:", accel_windows.shape)         # (n_windows, 51200)
print("Accel df shape:", df_accel_win.shape)
print("Accel is_leak counts:", df_accel_win['is_leak'].value_counts())


In [ ]:
df_accel = df_accel[df_accel['flow'] != 'Transient'].reset_index(drop=True)

In [ ]:
import numpy as np
import librosa
from scipy import stats
from scipy.signal import butter, sosfiltfilt

def butter_bandpass_sos(lowcut, highcut, fs, order=4):
    """Design a Butterworth bandpass filter, return as SOS (numerically stable)."""
    nyq = fs / 2
    low = lowcut / nyq
    high = highcut / nyq
    sos = butter(order, [low, high], btype='band', output='sos')
    return sos

def apply_bandpass(signal, fs, lowcut, highcut, order=4):
    """Apply zero-phase Butterworth bandpass filter."""
    sos = butter_bandpass_sos(lowcut, highcut, fs, order)
    return sosfiltfilt(sos, signal)

def extract_features(window, sr, bandpass=None):
    """Extract a feature vector from a single window.

    bandpass: tuple (lowcut, highcut) in Hz, or None for no filtering.
    """
    # === Apply Butterworth bandpass filter if requested ===
    if bandpass is not None:
        lowcut, highcut = bandpass
        window = apply_bandpass(window, sr, lowcut, highcut, order=4)
        window = window.astype(np.float32)

    feats = {}

    # === Statistical features (time domain) ===
    feats['rms'] = np.sqrt(np.mean(window**2))
    feats['mean_abs'] = np.mean(np.abs(window))
    feats['std'] = np.std(window)
    feats['kurtosis'] = stats.kurtosis(window)
    feats['skew'] = stats.skew(window)
    feats['peak'] = np.max(np.abs(window))
    feats['crest_factor'] = feats['peak'] / (feats['rms'] + 1e-10)
    feats['zcr'] = np.mean(np.abs(np.diff(np.sign(window)))) / 2

    # === Spectral features ===
    n_fft = min(2048, len(window))
    hop = n_fft // 4

    spec_centroid = librosa.feature.spectral_centroid(y=window, sr=sr, n_fft=n_fft, hop_length=hop)
    feats['spec_centroid_mean'] = np.mean(spec_centroid)
    feats['spec_centroid_std'] = np.std(spec_centroid)

    spec_bw = librosa.feature.spectral_bandwidth(y=window, sr=sr, n_fft=n_fft, hop_length=hop)
    feats['spec_bw_mean'] = np.mean(spec_bw)

    spec_rolloff = librosa.feature.spectral_rolloff(y=window, sr=sr, n_fft=n_fft, hop_length=hop)
    feats['spec_rolloff_mean'] = np.mean(spec_rolloff)

    spec_flatness = librosa.feature.spectral_flatness(y=window, n_fft=n_fft, hop_length=hop)
    feats['spec_flatness_mean'] = np.mean(spec_flatness)

    # === Band energies ===
    fft = np.fft.rfft(window)
    power = np.abs(fft)**2
    freqs = np.fft.rfftfreq(len(window), 1/sr)
    total_power = np.sum(power) + 1e-10

    nyq = sr / 2
    bands = [(0, 0.05), (0.05, 0.15), (0.15, 0.3), (0.3, 0.5), (0.5, 0.75), (0.75, 1.0)]
    for i, (lo, hi) in enumerate(bands):
        mask = (freqs >= lo*nyq) & (freqs < hi*nyq)
        feats[f'band_{i}_energy'] = np.sum(power[mask]) / total_power

    # === MFCCs ===
    n_mfcc = 13
    mfccs = librosa.feature.mfcc(y=window, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop)
    for i in range(n_mfcc):
        feats[f'mfcc_{i}_mean'] = np.mean(mfccs[i])
        feats[f'mfcc_{i}_std'] = np.std(mfccs[i])

    return feats

def extract_features_batch(windows, sr, bandpass=None):
    """Extract features for all windows. Returns a DataFrame."""
    rows = []
    for w in windows:
        rows.append(extract_features(w, sr, bandpass=bandpass))
    return pd.DataFrame(rows)

In [ ]:
ACCEL_BAND = (100, 1000)    # Hz
accel_feats_filt = extract_features_batch(accel_windows, SR_ACCEL, bandpass=ACCEL_BAND)
print("Accel filtered:", accel_feats_filt.shape, "NaN:", accel_feats_filt.isna().sum().sum())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, recall_score, confusion_matrix, classification_report
import lightgbm as lgb

def evaluate_modality_1(features_df, meta_df, modality_name, n_splits=5, threshold=0.5):
    """Train + evaluate a LightGBM detector with grouped stratified k-fold CV."""
    X = features_df.values
    y = meta_df['is_leak'].values
    groups = meta_df['signal_id'].values

    # StratifiedGroupKFold: stratify on label, group by signal_id
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_metrics = []
    all_y_true = []
    all_y_pred = []
    all_y_proba = []

    for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups)):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        # Class imbalance handling via scale_pos_weight
        n_neg = (y_tr == 0).sum()
        n_pos = (y_tr == 1).sum()
        scale = n_neg / n_pos  # < 1 since leak is majority — actually we want neg upweighted

        model = lgb.LGBMClassifier(
            n_estimators=300,
            learning_rate=0.1,
            num_leaves=35,
            class_weight='balanced',  # handles imbalance for us
            random_state=42,
            verbose=-1,
        )
        model.fit(X_tr, y_tr)

        y_proba = model.predict_proba(X_te)[:, 1]
        y_pred = (y_proba >= threshold).astype(int)

        # Window-level metrics
        f1 = f1_score(y_te, y_pred)
        sensitivity = recall_score(y_te, y_pred, pos_label=1)  # leak recall
        specificity = recall_score(y_te, y_pred, pos_label=0)  # no-leak recall

        fold_metrics.append({
            'fold': fold, 'f1': f1, 'sensitivity': sensitivity, 'specificity': specificity,
            'n_train': len(train_idx), 'n_test': len(test_idx)
        })
        all_y_true.extend(y_te); all_y_pred.extend(y_pred); all_y_proba.extend(y_proba)

    metrics_df = pd.DataFrame(fold_metrics)
    print(f"\n=== {modality_name} ===")
    print(metrics_df.to_string(index=False))
    print(f"\nMean: F1={metrics_df['f1'].mean():.3f} (±{metrics_df['f1'].std():.3f}), "
          f"Sens={metrics_df['sensitivity'].mean():.3f} (±{metrics_df['sensitivity'].std():.3f}), "
          f"Spec={metrics_df['specificity'].mean():.3f} (±{metrics_df['specificity'].std():.3f})")
    print("\nOverall confusion matrix (window-level):")
    print(confusion_matrix(all_y_true, all_y_pred))

    return metrics_df, np.array(all_y_true), np.array(all_y_pred), np.array(all_y_proba), model


accel_metrics, accel_yt, accel_yp, accel_yprob, accel_model = evaluate_modality_1(
    accel_feats_filt, df_accel_win, "ACCELEROMETER"
)